# Hydraulic Conductivity Workflow — Organized Version

This notebook reorganizes the full HK workflow into a consistent structure:

1. **Imports** in one place  
2. **Inputs, outputs, and constants** in one place  
3. **Helpers** in one place  
4. **Processing stages** grouped in execution order  
5. **NumPy-style docstrings** for reusable functions

The processing logic follows the original notebook, but the layout is cleaner and easier to maintain.


In [ ]:
"""Hydraulic-conductivity processing workflow for the Great Lakes model.

This notebook organizes the HK workflow into a single import section,
a single configuration section, a single helper-function section,
and ordered execution cells.

Notes
-----
- The code assumes ArcGIS Pro / ArcPy with Spatial Analyst.
- Paths and constants are defined in the next cell.
- Reusable logic is wrapped in functions with NumPy-style docstrings.
"""

import math
import os
import re
from collections import Counter, defaultdict

import arcpy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from arcpy.sa import Con, IsNull, Log10, Nibble, Raster, SetNull
from matplotlib.collections import PatchCollection
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap
from matplotlib.patches import Polygon as MplPolygon
from rasterio.merge import merge

In [ ]:
# ------------------------------------------------------------------
# INPUTS, OUTPUTS, AND CONSTANTS
# ------------------------------------------------------------------
CONFIG = {
    "surfacegeology": r"S:\Data\Other_Data\Xu_2021\Data\09_Integrated_Surficial_Geology_Map\Integrated_surficial_geology_mapping.shp",
    "excel_path": r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\Calibrated_surficial_Kh.xlsx",
    "in_sheet": r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\Calibrated_surficial_Kh.xlsx\GLB_surf_dissolve_merged$",
    "extended_buffer_2km": r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extended_Bdry_final_GLB_Albers_exported.shp",
    "water_mask_raster": r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes_ras.tif",
    "template_raster": r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_extended20kmbdr_1000m.tif",
    "lake_shp": r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes.shp",
    "mi_krig_up": r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\UP_hkBulk_3174.tif",
    "mi_krig_lp": r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\LP_hkBulk_3174.tif",
    "model_bdry": r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extended_Bdry_final_GLB_Albers_exported.shp",
    "out_dir": r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK",
    "gdb_name": "HK_join_clip.gdb",
    "merged_krig_name": "HKBUlk_MI_Krig.tif",
    "final_polygon_name": "Modified_HK_litkrig.shp",
    "multiband_hk_name": "HK_5band_1000m.tif",
    "multiband_plot_name": "HK_5band_plot.png",
    "sec_per_day": 86400.0,
    "lake_raster_value": 1,
    "lower_default_ms": 1e-7,
    "frac_bedrock_ms": 1e-6,
    "bedrock_ms": 1e-10,
    "lake_value": 100.0,
    "land_value": 0.0,
    "min_overlap_fraction": 0.05,
    "min_lake_overlap_fraction": 0.50,
    "tolerance": 0.30,
    "mi_krig_source_epsg": 4269,
    "fields_5_layers": ["UPKh_m_d", "MidKh_m_d", "LowKh_m_d", "FracKh_m_d", "BedKh_m_d"],
    "fields_5_layers_final": ["UPKh_krig", "MidKh_krig", "LowKh_m_d", "FracKh_m_d", "BedKh_m_d"],
    "hk_colors_15": [
        "#08088B", "#0D0887", "#2D0594", "#5C01A6", "#7E03A8",
        "#9C179E", "#BD3786", "#D8576B", "#E8765C", "#ED7953",
        "#F89441", "#FB9F3A", "#FDCA26", "#F0F921", "#FCFFA4",
    ],
}

CONFIG["gdb"] = os.path.join(CONFIG["out_dir"], CONFIG["gdb_name"])
CONFIG["frac_bedrock_mday"] = CONFIG["frac_bedrock_ms"] * CONFIG["sec_per_day"]
CONFIG["bedrock_mday"] = CONFIG["bedrock_ms"] * CONFIG["sec_per_day"]
CONFIG["merged_krig_path"] = os.path.join(CONFIG["out_dir"], CONFIG["merged_krig_name"])
CONFIG["final_polygon_path"] = os.path.join(CONFIG["out_dir"], CONFIG["final_polygon_name"])
CONFIG["multiband_hk_path"] = os.path.join(CONFIG["out_dir"], CONFIG["multiband_hk_name"])
CONFIG["multiband_plot_path"] = os.path.join(CONFIG["out_dir"], CONFIG["multiband_plot_name"])

os.makedirs(CONFIG["out_dir"], exist_ok=True)

if not arcpy.Exists(CONFIG["gdb"]):
    arcpy.management.CreateFileGDB(CONFIG["out_dir"], CONFIG["gdb_name"])

print("Configuration loaded.")
print(f"Output directory: {CONFIG['out_dir']}")
print(f"Workspace GDB   : {CONFIG['gdb']}")

## Run order

Run the cells in this order:

1. Imports  
2. Configuration  
3. Helpers  
4. Stage 1 through Stage 7

If you want to change any file path or output name, update only the **configuration** cell.


In [ ]:
# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------
def setup_arcpy_environment(overwrite_output=True):
    """Configure ArcPy for the HK workflow.

    Parameters
    ----------
    overwrite_output : bool, default=True
        If ``True``, ArcPy is allowed to overwrite existing outputs.

    Returns
    -------
    None
        The function updates ArcPy's global environment in place.
    """
    arcpy.CheckOutExtension("Spatial")
    arcpy.env.overwriteOutput = overwrite_output


def delete_if_exists(path):
    """Delete a dataset if it already exists.

    Parameters
    ----------
    path : str
        ArcPy-readable path to a feature class, raster, table, or folder.

    Returns
    -------
    None
        The function deletes ``path`` only when it exists.
    """
    if path and arcpy.Exists(path):
        arcpy.management.Delete(path)


def ensure_field(dataset, name, field_type, length=None):
    """Add a field to a dataset when the field is missing.

    Parameters
    ----------
    dataset : str
        Path to a feature class or table.
    name : str
        Field name to create.
    field_type : str
        ArcGIS field type such as ``"TEXT"`` or ``"DOUBLE"``.
    length : int, optional
        Text-field length. Used only when ``field_type`` is ``"TEXT"``.

    Returns
    -------
    None
        The function modifies the input dataset in place.
    """
    existing = {field.name for field in arcpy.ListFields(dataset)}
    if name not in existing:
        if field_type.upper() == "TEXT" and length is not None:
            arcpy.management.AddField(dataset, name, field_type, field_length=length)
        else:
            arcpy.management.AddField(dataset, name, field_type)


def make_join_id(dataset, source_field, join_field="JOIN_ID"):
    """Create a normalized text join key from an existing field.

    Parameters
    ----------
    dataset : str
        Input table or feature class.
    source_field : str
        Field whose values will be converted to a join-safe key.
    join_field : str, default="JOIN_ID"
        Name of the output key field.

    Returns
    -------
    None
        The function writes the normalized join key to ``join_field``.
    """
    ensure_field(dataset, join_field, "TEXT", length=50)
    codeblock = """def to_key(v):
        if v is None:
            return None
        s = str(v).strip()
        if s == "":
            return None
        try:
            fv = float(s)
            if fv.is_integer():
                s = str(int(fv))
            else:
                s = str(fv)
        except Exception:
            pass
        return s.strip().upper()"""
    arcpy.management.CalculateField(
        dataset,
        join_field,
        f"to_key(!{source_field}!)",
        "PYTHON3",
        code_block=codeblock,
    )


def safe_clip(in_fc, clip_fc, out_fc):
    """Clip features using the best available ArcPy tool.

    Parameters
    ----------
    in_fc : str
        Input feature class.
    clip_fc : str
        Clip feature class.
    out_fc : str
        Output feature class.

    Returns
    -------
    None
        The clipped output is written to ``out_fc``.
    """
    delete_if_exists(out_fc)
    if hasattr(arcpy.analysis, "PairwiseClip"):
        arcpy.analysis.PairwiseClip(in_fc, clip_fc, out_fc)
    else:
        arcpy.analysis.Clip(in_fc, clip_fc, out_fc)


def safe_erase(in_fc, erase_fc, out_fc):
    """Erase features using the best available ArcPy tool.

    Parameters
    ----------
    in_fc : str
        Input feature class.
    erase_fc : str
        Erase feature class.
    out_fc : str
        Output feature class.

    Returns
    -------
    None
        The erased output is written to ``out_fc``.
    """
    delete_if_exists(out_fc)
    if hasattr(arcpy.analysis, "PairwiseErase"):
        arcpy.analysis.PairwiseErase(in_fc, erase_fc, out_fc)
    else:
        arcpy.analysis.Erase(in_fc, erase_fc, out_fc)


def get_base_description(desc):
    """Remove thickness or subtype qualifiers from a sediment description.

    Parameters
    ----------
    desc : str or None
        Original polygon description.

    Returns
    -------
    str or None
        Simplified base description for fuzzy matching.
    """
    if desc is None:
        return None

    qualifiers = [
        r",\s*(thick|thin|discontinuous)$",
        r"\s*-\s*(Blanket|Veneer|Moraine complex)$",
        r"\s*-\s*(Undifferentiated sediments|Undifferentiated deposits|Undifferentiated)$",
        r"\s*-\s*(Ice-contact sediments|Outwash plain sediments)$",
        r"\s*-\s*(Littoral and nearshore sediments|Offshore sediments)$",
    ]

    base = desc.strip()
    for pattern in qualifiers:
        base = re.sub(pattern, "", base, flags=re.IGNORECASE)

    return base.strip()


def geometric_mean_range(kmin, kmax):
    """Compute the geometric mean of a conductivity range.

    Parameters
    ----------
    kmin : float
        Lower conductivity bound.
    kmax : float
        Upper conductivity bound.

    Returns
    -------
    float
        Geometric mean of ``kmin`` and ``kmax``.
    """
    return 10 ** ((math.log10(kmin) + math.log10(kmax)) / 2)


def format_tick(value):
    """Format conductivity values for plot colorbars.

    Parameters
    ----------
    value : float
        Conductivity value to format.

    Returns
    -------
    str
        Human-readable label string.
    """
    if value >= 100:
        return f"{value:.0f}"
    if value >= 10:
        return f"{value:.1f}"
    if value >= 1:
        return f"{value:.2f}"
    if value >= 0.01:
        return f"{value:.3f}"
    return f"{value:.1e}"


def compute_quantile_bounds(records, key1, key2, n_classes):
    """Compute shared quantile boundaries for two value fields.

    Parameters
    ----------
    records : list of dict
        Polygon records holding geometry paths and HK values.
    key1 : str
        First field name.
    key2 : str
        Second field name.
    n_classes : int
        Number of desired quantile classes.

    Returns
    -------
    tuple of (numpy.ndarray, numpy.ndarray)
        Quantile boundaries and all finite values used to build them.
    """
    values = []
    for record in records:
        for key in (key1, key2):
            value = record[key]
            if value is not None and value > 0.001:
                values.append(value)

    values = np.array(sorted(values), dtype=float)
    percentiles = np.linspace(0, 100, n_classes + 1)
    bounds = np.percentile(values, percentiles)
    bounds = np.unique(bounds)
    return bounds, values


def plot_hk_map_quantile(ax, records, value_key, title, bounds, cmap):
    """Plot polygon HK values using shared quantile classes.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Target axis for plotting.
    records : list of dict
        Polygon records containing geometry coordinates and values.
    value_key : str
        Value field to render.
    title : str
        Panel title.
    bounds : array-like
        Quantile boundaries used for classification.
    cmap : matplotlib.colors.Colormap
        Colormap for rendering.

    Returns
    -------
    tuple
        Patch collection and normalization object.
    """
    patches = []
    values = []

    for record in records:
        value = record[value_key]
        if value is None or value <= 0:
            value = bounds[0]
        for coords in record["paths"]:
            patches.append(MplPolygon(coords, closed=True))
            values.append(value)

    values = np.array(values, dtype=float)
    values = np.clip(values, bounds[0], bounds[-1])
    norm = BoundaryNorm(bounds, cmap.N)

    patch_collection = PatchCollection(
        patches,
        cmap=cmap,
        norm=norm,
        edgecolor="face",
        linewidth=0,
        antialiased=False,
        rasterized=True,
    )
    patch_collection.set_array(values)

    ax.add_collection(patch_collection)
    ax.autoscale()
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12, color="black")
    ax.ticklabel_format(style="plain", axis="both")
    ax.tick_params(labelsize=8, colors="black")
    ax.set_xlabel("Easting (m)", fontsize=10, color="black")
    ax.set_ylabel("Northing (m)", fontsize=10, color="black")
    ax.set_facecolor("white")

    for spine in ax.spines.values():
        spine.set_color("black")

    return patch_collection, norm


LITERATURE_RANGES = {
    "coastal_sand":               (0.086,     86.0),
    "colluvial":                  (0.0864,    8.64),
    "glaciolacustrine_offshore":  (0.0000864, 0.0864),
    "glaciolacustrine_nearshore": (0.00864,   8.64),
    "lacustrine_nearshore":       (0.00864,   8.64),
    "lacustrine_offshore":        (0.0000864, 0.0864),
    "silt_sandy_clayey_till":     (0.0000864, 0.0864),
    "glacial_till":               (8.64e-8,   0.0864),
    "glaciofluvial":              (0.864,     86.4),
    "silty_sand":                 (0.00864,   86.4),
    "peat_organic":               (0.0864,    8.64),
    "sandstone":                  (8.64e-6,   0.432),
    "glaciomarine_nearshore":     (0.00864,   8.64),
    "glaciomarine_offshore":      (0.0000864, 0.0864),
    "volcanic":                   (8.64e-6,   0.432),
}

DESCRIPTION_TO_MATERIAL = {
    "Coastal zone sediments, mostly medium-grained":                 "coastal_sand",
    "Colluvial sediments and residual material":                     "colluvial",
    "Colluvial sediments, thin":                                     "colluvial",
    "Colluvial sediments, discontinuous":                            "colluvial",
    "Colluvial and alluvial sediments":                              "colluvial",
    "Glaciolacustrine sediments - Offshore sediments":               "glaciolacustrine_offshore",
    "Glaciolacustrine sediments - Littoral and nearshore sediments": "glaciolacustrine_nearshore",
    "Lacustrine sediments - Littoral and nearshore sediments":       "lacustrine_nearshore",
    "Lacustrine sediments - Offshore sediments":                     "lacustrine_offshore",
    "Glacial sediments - Blanket":                                   "silt_sandy_clayey_till",
    "Glacial sediments - Veneer":                                    "silt_sandy_clayey_till",
    "Glacial sediments - Moraine complex":                           "glacial_till",
    "Glaciofluvial sediments - Ice-contact sediments":               "glaciofluvial",
    "Glaciofluvial sediments - Outwash plain sediments":             "glaciofluvial",
    "Glacial till sediments, mostly sandy, thin":                    "silty_sand",
    "Glacial till sediments, mostly sandy, discontinuous":           "silty_sand",
    "Organic deposits - Undifferentiated deposits":                  "peat_organic",
    "Bedrock - Undifferentiated":                                    "sandstone",
    "Glaciomarine sediments - Littoral and nearshore sediments":     "glaciomarine_nearshore",
    "Glaciomarine sediments - Offshore sediments":                   "glaciomarine_offshore",
    "Volcanic deposits - Undifferentiated":                          "volcanic",
}

In [ ]:
# ------------------------------------------------------------------
# STAGE 1 — JOIN CALIBRATED HK ATTRIBUTES AND BUILD INITIAL POLYGONS
# ------------------------------------------------------------------
setup_arcpy_environment()

surf_fc = os.path.join(CONFIG["gdb"], "surfacegeology_base")
csv_tbl = os.path.join(CONFIG["gdb"], "Kh_csv_tbl")
joined_fc = os.path.join(CONFIG["gdb"], "surfacegeo_joined")

# Copy the source geology layer into geodatabase, Import the calibrated HK CSV, 
# Standardize the join keys, and join the conductivity attributes to the geology polygons.

print("Copying geology to the workspace geodatabase...")
delete_if_exists(surf_fc)
arcpy.management.CopyFeatures(CONFIG["surfacegeology"], surf_fc)

print("Importing conductivity table...")
delete_if_exists(csv_tbl)
arcpy.conversion.TableToTable(CONFIG["in_sheet"], CONFIG["gdb"], "Kh_csv_tbl")

table_fields = [field.name for field in arcpy.ListFields(csv_tbl)]
if "Lower_ms" not in table_fields:
    ensure_field(csv_tbl, "Lower_ms", "DOUBLE")
    arcpy.management.CalculateField(csv_tbl, "Lower_ms", str(CONFIG["lower_default_ms"]), "PYTHON3")

print("Creating join keys...")
make_join_id(surf_fc, "POLYID", "JOIN_ID")
make_join_id(csv_tbl, "Zone_", "JOIN_ID")

print("Joining calibrated HK attributes...")
delete_if_exists(joined_fc)
arcpy.management.CopyFeatures(surf_fc, joined_fc)
csv_fields_to_add = [
    field.name
    for field in arcpy.ListFields(csv_tbl)
    if field.type not in ("OID", "Geometry") and field.name.lower() != "join_id"
]
arcpy.management.JoinField(joined_fc, "JOIN_ID", csv_tbl, "JOIN_ID", csv_fields_to_add)

print("Calculating five HK layers in m/day...")
for field_name in CONFIG["fields_5_layers"]:
    ensure_field(joined_fc, field_name, "DOUBLE")

# Calculate the five HK layers in m/day, using the source fields from the joined CSV. 
# The source fields are expected to be in m/s and named "Kh__m_s_",
# "Kh__m_s_1", and "Lower_ms". If any of the source fields are missing, raise an error.

joined_fields = [field.name for field in arcpy.ListFields(joined_fc)]
upper_src = "Kh__m_s_"
middle_src = "Kh__m_s_1"
lower_src = "Lower_ms"

missing = [field_name for field_name in (upper_src, middle_src, lower_src) if field_name not in joined_fields]
if missing:
    raise ValueError(f"Missing expected source field(s): {missing}")

codeblock = """
def to_float(v):
    try:
        if v is None:
            return None
        s = str(v).strip()
        if s == "":
            return None
        return float(s)
    except Exception:
        return None
"""

arcpy.management.CalculateField(
    joined_fc,
    "UPKh_m_d",
    f"to_float(!{upper_src}!) * {CONFIG['sec_per_day']}",
    "PYTHON3",
    codeblock,
)
arcpy.management.CalculateField(
    joined_fc,
    "MidKh_m_d",
    f"to_float(!{middle_src}!) * {CONFIG['sec_per_day']}",
    "PYTHON3",
    codeblock,
)
arcpy.management.CalculateField(
    joined_fc,
    "LowKh_m_d",
    f"to_float(!{lower_src}!) * {CONFIG['sec_per_day']}",
    "PYTHON3",
    codeblock,
)
arcpy.management.CalculateField(joined_fc, "FracKh_m_d", str(CONFIG["frac_bedrock_mday"]), "PYTHON3")
arcpy.management.CalculateField(joined_fc, "BedKh_m_d", str(CONFIG["bedrock_mday"]), "PYTHON3")

## Clips the geology/HK polygons to the buffered model domain, 
# extracts lake polygons from the water mask, splits land and lake portions, and assigns lake HK values where needed.

print("Clipping geology polygons to the model domain and applying lake values...")
domain_fc = os.path.join(CONFIG["gdb"], "domain_buffer2km_proj")
clip_fc = os.path.join(CONFIG["gdb"], "surfacegeo_joined_clip2km")
lake_poly0 = os.path.join(CONFIG["gdb"], "lake_poly0")
lake_poly = os.path.join(CONFIG["gdb"], "lake_diss")
lake_domain = os.path.join(CONFIG["gdb"], "lake_in_domain")
lake_part = os.path.join(CONFIG["gdb"], "hk_lake_part")
land_part = os.path.join(CONFIG["gdb"], "hk_land_part")
lake_missing = os.path.join(CONFIG["gdb"], "hk_lake_missing")
final_fc = os.path.join(CONFIG["gdb"], "surfacegeo_HK_FINAL")

buffer_sr = arcpy.Describe(CONFIG["extended_buffer_2km"]).spatialReference
geology_sr = arcpy.Describe(joined_fc).spatialReference
delete_if_exists(domain_fc)

if geology_sr.factoryCode == buffer_sr.factoryCode:
    arcpy.management.CopyFeatures(CONFIG["extended_buffer_2km"], domain_fc)
else:
    arcpy.management.Project(CONFIG["extended_buffer_2km"], domain_fc, geology_sr)

delete_if_exists(clip_fc)
safe_clip(joined_fc, domain_fc, clip_fc)

for dataset in (lake_poly0, lake_poly, lake_domain, lake_part, land_part, lake_missing, final_fc):
    delete_if_exists(dataset)

arcpy.conversion.RasterToPolygon(CONFIG["water_mask_raster"], lake_poly0, "NO_SIMPLIFY", "VALUE")
arcpy.management.Dissolve(lake_poly0, lake_poly)

safe_clip(lake_poly, domain_fc, lake_domain)
safe_clip(clip_fc, lake_domain, lake_part)
safe_erase(clip_fc, lake_domain, land_part)
safe_erase(lake_domain, lake_part, lake_missing)

for feature_class in (lake_part, land_part, lake_missing):
    ensure_field(feature_class, "Rech_m_d", "DOUBLE")

arcpy.management.CalculateField(land_part, "Rech_m_d", str(CONFIG["land_value"]), "PYTHON3")
arcpy.management.CalculateField(lake_part, "Rech_m_d", str(CONFIG["lake_value"]), "PYTHON3")
arcpy.management.CalculateField(lake_missing, "Rech_m_d", str(CONFIG["lake_value"]), "PYTHON3")

for field_name in ("UPKh_m_d", "MidKh_m_d", "LowKh_m_d"):
    ensure_field(lake_part, field_name, "DOUBLE")
    ensure_field(lake_missing, field_name, "DOUBLE")
    arcpy.management.CalculateField(lake_part, field_name, str(CONFIG["lake_value"]), "PYTHON3")
    arcpy.management.CalculateField(lake_missing, field_name, str(CONFIG["lake_value"]), "PYTHON3")

for field_name, value in (("FracKh_m_d", CONFIG["frac_bedrock_mday"]), ("BedKh_m_d", CONFIG["bedrock_mday"])):
    ensure_field(lake_part, field_name, "DOUBLE")
    ensure_field(lake_missing, field_name, "DOUBLE")
    arcpy.management.CalculateField(lake_part, field_name, str(value), "PYTHON3")
    arcpy.management.CalculateField(lake_missing, field_name, str(value), "PYTHON3")

arcpy.management.Merge([land_part, lake_part, lake_missing], final_fc)

print(f"Initial HK polygon feature class ready: {final_fc}")

In [ ]:
# ------------------------------------------------------------------
# STAGE 2 — MERGE MICHIGAN KRIGING RASTERS
# ------------------------------------------------------------------
# 1- add two columns of called KrigUP_HK and KrigMid_HK
# 2- Do a zonal stats of this final_fc = os.path.join(gdb, "surfacegeo_HK_FINAL") with these two rasters
# MIKRIG_UP = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\UP_hkBulk_3174.tif"
# MIKRIG_LP = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\LP_hkBulk_3174.tif" and 
# replace the UPKh_m_d of those overlapping area with the rasters in the KrigUP_HK column 


print("Merging Michigan kriging rasters at 1000 m resolution...")

src_files = [rasterio.open(CONFIG["mi_krig_up"]), rasterio.open(CONFIG["mi_krig_lp"])]
mosaic, out_transform = merge(src_files, res=(1000, 1000), method="first")

out_meta = src_files[0].meta.copy()
out_meta.update(
    {
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_transform,
        "count": mosaic.shape[0],
        "crs": src_files[0].crs,
        "dtype": mosaic.dtype,
    }
)

with rasterio.open(CONFIG["merged_krig_path"], "w", **out_meta) as destination:
    destination.write(mosaic)

for src in src_files:
    src.close()

print(f"Merged kriging raster saved to: {CONFIG['merged_krig_path']}")

In [ ]:
# ------------------------------------------------------------------
# STAGE 3 — ASSIGN KRIGING VALUES TO POLYGONS
# ------------------------------------------------------------------
# extracts hydraulic conductivity (HK) values from a Michigan Basin kriging raster and 
# Assigns them to Great Lakes Basin polygons using the **geometric mean** approach:`UPKh_krig = 10^(mean(log10(HK)))` per polygon

final_fc = os.path.join(CONFIG["gdb"], "surfacegeo_HK_FINAL")
poly_3174 = os.path.join(CONFIG["gdb"], "poly_3174")
raster_3174 = os.path.join(CONFIG["gdb"], "MIKrig_3174")
raster_domain_raw = os.path.join(CONFIG["gdb"], "temp_domain_raw")
raster_domain = os.path.join(CONFIG["gdb"], "temp_domain")
clipped_fc_raw = os.path.join(CONFIG["gdb"], "surfacegeo_HK_Krig_clipped_raw")
clipped_fc = os.path.join(CONFIG["gdb"], "surfacegeo_HK_Krig_clipped")
log_raster_path = os.path.join(CONFIG["gdb"], "temp_log10_HK")
zs_table = os.path.join(CONFIG["gdb"], "temp_zs_krig")

# ============================================================
# STEP 1. Reproject POLYGON to EPSG:3174
# ============================================================
print("Reprojecting polygon and raster to EPSG:3174 when needed...")
sr_3174 = arcpy.SpatialReference(3174)

poly_sr = arcpy.Describe(final_fc).spatialReference
delete_if_exists(poly_3174)
if poly_sr.factoryCode != 3174:
    arcpy.management.Project(final_fc, poly_3174, sr_3174)
    print(f"Polygon projected to: {poly_3174}")
else:
    poly_3174 = final_fc
    print("Polygon already in EPSG:3174.")

ras_sr = arcpy.Describe(CONFIG["merged_krig_path"]).spatialReference
in_coor_system = None if ras_sr.factoryCode != 0 else arcpy.SpatialReference(CONFIG["mi_krig_source_epsg"])

delete_if_exists(raster_3174)
if ras_sr.factoryCode != 3174:
    arcpy.management.ProjectRaster(
        CONFIG["merged_krig_path"],
        raster_3174,
        sr_3174,
        "BILINEAR",
        in_coor_system=in_coor_system,
    )
    MIKrig_work = raster_3174
    print(f"Raster projected to: {raster_3174}")
else:
    MIKrig_work = CONFIG["merged_krig_path"]
    print("Raster already in EPSG:3174.")

# ============================================================
# STEP 2.  Build raster domain polygon, clip geology polygons to it, and remove slivers
# ============================================================
print("Building raster domain polygon...")
delete_if_exists(raster_domain_raw)
delete_if_exists(raster_domain)

ras_obj = Raster(MIKrig_work)
valid_mask = Con(~IsNull(ras_obj), 1)
arcpy.conversion.RasterToPolygon(valid_mask, raster_domain_raw, "NO_SIMPLIFY", "Value", "SINGLE_OUTER_PART")
arcpy.management.Dissolve(raster_domain_raw, raster_domain)
delete_if_exists(raster_domain_raw)

# ============================================================
# Add unique join key BEFORE clipping
# ============================================================

print("Adding unique polygon IDs before clipping...")
ensure_field(poly_3174, "UNIQ_ID", "LONG")
arcpy.management.CalculateField(poly_3174, "UNIQ_ID", f"!{arcpy.Describe(poly_3174).OIDFieldName}!", "PYTHON3")

#Store original areas for sliver detection later

orig_areas = {}
with arcpy.da.SearchCursor(poly_3174, ["UNIQ_ID", "SHAPE@AREA"]) as cursor:
    for uniq_id, area in cursor:
        orig_areas[uniq_id] = area
        
# ============================================================
# STEP 4. Clip polygon to raster domain + remove slivers
# ============================================================

print("Clipping polygons to raster domain and removing slivers...")
delete_if_exists(clipped_fc_raw)
delete_if_exists(clipped_fc)
arcpy.analysis.Clip(poly_3174, raster_domain, clipped_fc_raw)

oid_field_raw = arcpy.Describe(clipped_fc_raw).OIDFieldName
uniq_fragments = defaultdict(list)
with arcpy.da.SearchCursor(clipped_fc_raw, [oid_field_raw, "UNIQ_ID", "SHAPE@AREA"]) as cursor:
    for oid, uniq_id, area in cursor:
        uniq_fragments[uniq_id].append((oid, area))

keep_oids = set()
removed_slivers = []
for uniq_id, fragments in uniq_fragments.items():
    total_clip_area = sum(area for _, area in fragments)
    orig_area = orig_areas.get(uniq_id, 0)
    if orig_area > 0 and (total_clip_area / orig_area) < CONFIG["min_overlap_fraction"]:
        removed_slivers.append(uniq_id)
        continue
    best_oid = max(fragments, key=lambda item: item[1])[0]
    keep_oids.add(best_oid)

all_oids = set()
with arcpy.da.SearchCursor(clipped_fc_raw, [oid_field_raw]) as cursor:
    for row in cursor:
        all_oids.add(row[0])

remove_oids = all_oids - keep_oids
if remove_oids:
    layer = "clipped_raw_lyr"
    arcpy.management.MakeFeatureLayer(clipped_fc_raw, layer)
    where = f"{oid_field_raw} IN ({','.join(str(oid) for oid in remove_oids)})"
    arcpy.management.SelectLayerByAttribute(layer, "NEW_SELECTION", where)
    arcpy.management.DeleteRows(layer)
    arcpy.management.Delete(layer)

arcpy.management.CopyFeatures(clipped_fc_raw, clipped_fc)
delete_if_exists(clipped_fc_raw)

# ============================================================
# STEP 5. Add ZONE_ID, UPKh_krig, MidKh_krig to clipped FC
# ============================================================

print("Adding zonal-statistics fields...")
ensure_field(clipped_fc, "ZONE_ID", "LONG")
ensure_field(clipped_fc, "UPKh_krig", "DOUBLE")
ensure_field(clipped_fc, "MidKh_krig", "DOUBLE")
oid_field = arcpy.Describe(clipped_fc).OIDFieldName
arcpy.management.CalculateField(clipped_fc, "ZONE_ID", f"!{oid_field}!", "PYTHON3")

# ============================================================
# STEP 5b. Create log10-transformed raster
# ============================================================

print("Creating log10-transformed raster...")
delete_if_exists(log_raster_path)
log_raster = Con(Raster(MIKrig_work) > 0, Log10(Raster(MIKrig_work)))
log_raster.save(log_raster_path)

# ============================================================
# STEP 6. Zonal Statistics (MEAN) on log10(HK) raster
# ============================================================
print("Running zonal statistics on the log-transformed raster...")
arcpy.env.snapRaster = MIKrig_work
arcpy.env.cellSize = MIKrig_work
delete_if_exists(zs_table)
arcpy.sa.ZonalStatisticsAsTable(clipped_fc, "ZONE_ID", log_raster_path, zs_table, "DATA", "MEAN")
arcpy.env.snapRaster = None
arcpy.env.cellSize = None

zs_dict = {}
with arcpy.da.SearchCursor(zs_table, ["ZONE_ID", "MEAN", "COUNT"]) as cursor:
    for zone_id, mean_log_val, count_val in cursor:
        if mean_log_val is not None and count_val >= 1:
            zs_dict[zone_id] = 10 ** mean_log_val
            
# ============================================================
# STEP 7. Back-transform and write geometric mean to clipped FC
#         MidKh_krig = UPKh_krig (SAME VALUE)
# ============================================================
print("Writing geometric-mean kriging values to the clipped polygons...")
with arcpy.da.UpdateCursor(clipped_fc, ["ZONE_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
    for row in cursor:
        zone_id = row[0]
        if zone_id in zs_dict:
            value = zs_dict[zone_id]
            row[1] = value
            row[2] = value
            cursor.updateRow(row)
# ============================================================
# STEP 7b. Fill NULL polygons with nearest raster value
#          MidKh_krig = UPKh_krig (SAME VALUE)
# ============================================================
print("Filling remaining NULL kriging polygons with nearest raster values...")
null_zone_ids = set()
with arcpy.da.SearchCursor(clipped_fc, ["ZONE_ID", "UPKh_krig"]) as cursor:
    for zone_id, value in cursor:
        if value is None:
            null_zone_ids.add(zone_id)

if null_zone_ids:
    null_layer = "null_polys_lyr"
    null_centroids = os.path.join(CONFIG["gdb"], "temp_null_centroids")
    filled_raster = os.path.join(CONFIG["gdb"], "temp_filled_raster")
    sample_null = os.path.join(CONFIG["gdb"], "temp_sample_null")

    arcpy.management.MakeFeatureLayer(clipped_fc, null_layer)
    arcpy.management.SelectLayerByAttribute(
        null_layer,
        "NEW_SELECTION",
        f"ZONE_ID IN ({', '.join(str(zone_id) for zone_id in null_zone_ids)})",
    )
    delete_if_exists(null_centroids)
    arcpy.management.FeatureToPoint(null_layer, null_centroids, "INSIDE")
    arcpy.management.Delete(null_layer)

    ras_log = Raster(log_raster_path)
    valid_cells = Con(~IsNull(ras_log), 1)

    delete_if_exists(filled_raster)
    nibbled = Nibble(ras_log, valid_cells, "DATA_ONLY", "PROCESS_NODATA")
    nibbled.save(filled_raster)

    delete_if_exists(sample_null)
    arcpy.sa.Sample(filled_raster, null_centroids, sample_null, "NEAREST", "ZONE_ID")

    sample_fields = [field.name for field in arcpy.ListFields(sample_null)]
    link_field = sample_fields[1]
    raster_val_field = sample_fields[-1]

    centroid_oid_field = arcpy.Describe(null_centroids).OIDFieldName
    centroid_oid_to_zone = {}
    with arcpy.da.SearchCursor(null_centroids, [centroid_oid_field, "ZONE_ID"]) as cursor:
        for centroid_oid, zone_id in cursor:
            centroid_oid_to_zone[centroid_oid] = zone_id

    filled_values = {}
    with arcpy.da.SearchCursor(sample_null, [link_field, raster_val_field]) as cursor:
        for raw_oid, value in cursor:
            zone_id = (
                centroid_oid_to_zone.get(raw_oid)
                or centroid_oid_to_zone.get(int(raw_oid) if raw_oid is not None else None)
                or centroid_oid_to_zone.get(str(raw_oid))
            )
            if zone_id is not None and value is not None:
                filled_values[zone_id] = 10 ** value

    with arcpy.da.UpdateCursor(clipped_fc, ["ZONE_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
        for row in cursor:
            zone_id = row[0]
            if zone_id in filled_values:
                value = filled_values[zone_id]
                row[1] = value
                row[2] = value
                cursor.updateRow(row)

    delete_if_exists(null_centroids)
    delete_if_exists(sample_null)
    delete_if_exists(filled_raster)

print("Joining kriging values back to the original polygons...")
ensure_field(poly_3174, "UPKh_krig", "DOUBLE")
ensure_field(poly_3174, "MidKh_krig", "DOUBLE")

uniq_lookup = {}
sliver_ids = []
with arcpy.da.SearchCursor(clipped_fc, ["UNIQ_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
    for uniq_id, up_val, mid_val in cursor:
        if uniq_id is None or up_val is None:
            continue
        if uniq_id in uniq_lookup:
            sliver_ids.append(uniq_id)
            if up_val > uniq_lookup[uniq_id][0]:
                uniq_lookup[uniq_id] = (up_val, up_val)
        else:
            uniq_lookup[uniq_id] = (up_val, up_val)

with arcpy.da.UpdateCursor(poly_3174, ["UNIQ_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
    for row in cursor:
        uniq_id = row[0]
        if uniq_id in uniq_lookup:
            row[1] = uniq_lookup[uniq_id][0]
            row[2] = uniq_lookup[uniq_id][1]
        else:
            row[1] = None
            row[2] = None
        cursor.updateRow(row)

print(f"Kriging assignment complete. Working polygon layer: {poly_3174}")

In [ ]:
# ------------------------------------------------------------------
# STAGE 4 — FUZZY MATCHING, LITERATURE FALLBACK, AND MANUAL FIXES
# ------------------------------------------------------------------
# ============================================================
# Step 1 — Build Lookup Table from Michigan Polygons
# ============================================================
# Read all polygons that have UPKh_krig values (Michigan polygons)
# and create a reference table showing Description, POLYID, and HK values.
# Also store centroid coordinates for nearest-neighbor matching later.

print("Building Michigan lookup tables from polygons that already have kriging values...")
michigan_records = []
with arcpy.da.SearchCursor(
    poly_3174,
    ["UNIQ_ID", "POLYID", "Description", "UPKh_krig", "MidKh_krig", "SHAPE@"],
) as cursor:
    for uniq_id, polyid, desc, up_val, mid_val, shape in cursor:
        if up_val is not None:
            michigan_records.append(
                {
                    "UNIQ_ID": uniq_id,
                    "POLYID": polyid,
                    "Description": desc,
                    "UPKh_krig": up_val,
                    "MidKh_krig": mid_val,
                    "CENTROID_X": shape.centroid.X,
                    "CENTROID_Y": shape.centroid.Y,
                }
            )

michigan_df = pd.DataFrame(michigan_records)
null_records = []
with arcpy.da.SearchCursor(
    poly_3174,
    ["UNIQ_ID", "POLYID", "Description", "UPKh_krig", "SHAPE@"],
) as cursor:
    for uniq_id, polyid, desc, up_val, shape in cursor:
        if up_val is None:
            null_records.append(
                {
                    "UNIQ_ID": uniq_id,
                    "POLYID": polyid,
                    "Description": desc,
                    "CENTROID_X": shape.centroid.X,
                    "CENTROID_Y": shape.centroid.Y,
                }
            )

null_df = pd.DataFrame(null_records)
existing_vals = {}
with arcpy.da.SearchCursor(poly_3174, ["UNIQ_ID", "UPKh_m_d", "MidKh_m_d"]) as cursor:
    for uniq_id, up_val, mid_val in cursor:
        existing_vals[uniq_id] = (up_val, mid_val)

michigan_base_lookup = defaultdict(list)
for _, row in michigan_df.iterrows():
    base = get_base_description(row["Description"])
    michigan_base_lookup[base].append(
        {
            "POLYID": row["POLYID"],
            "Description": row["Description"],
            "UPKh_krig": row["UPKh_krig"],
            "MidKh_krig": row["MidKh_krig"],
            "x": row["CENTROID_X"],
            "y": row["CENTROID_Y"],
        }
    )

print("Running fuzzy matching for polygons still missing kriging values...")
fuzzy_assignments = {}
for _, null_row in null_df.iterrows():
    uniq_id = null_row["UNIQ_ID"]
    desc = null_row["Description"]
    base = get_base_description(str(desc))

    if base in michigan_base_lookup:
        null_x = null_row["CENTROID_X"]
        null_y = null_row["CENTROID_Y"]

        best_dist = float("inf")
        best_match = None
        for match_row in michigan_base_lookup[base]:
            dist = np.sqrt((null_x - match_row["x"]) ** 2 + (null_y - match_row["y"]) ** 2)
            if dist < best_dist:
                best_dist = dist
                best_match = match_row

        if best_match is not None:
            fuzzy_assignments[uniq_id] = {
                "UPKh_krig": best_match["UPKh_krig"],
                "MidKh_krig": best_match["MidKh_krig"],
                "source_POLYID": best_match["POLYID"],
                "source_desc": best_match["Description"],
                "distance_m": best_dist,
            }

with arcpy.da.UpdateCursor(poly_3174, ["UNIQ_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
    for row in cursor:
        uniq_id = row[0]
        if row[1] is None and uniq_id in fuzzy_assignments:
            row[1] = fuzzy_assignments[uniq_id]["UPKh_krig"]
            row[2] = fuzzy_assignments[uniq_id]["MidKh_krig"]
            cursor.updateRow(row)

print("Assigning literature-based or existing fallback values to any remaining NULL polygons...")
remaining_nulls = []
with arcpy.da.SearchCursor(
    poly_3174,
    ["UNIQ_ID", "POLYID", "Description", "UPKh_krig", "UPKh_m_d", "MidKh_m_d"],
) as cursor:
    for uniq_id, polyid, desc, up_krig, up_md, mid_md in cursor:
        if up_krig is None:
            remaining_nulls.append(
                {
                    "UNIQ_ID": uniq_id,
                    "POLYID": polyid,
                    "Description": str(desc),
                    "UPKh_m_d": up_md,
                    "MidKh_m_d": mid_md,
                }
            )

lit_assignments = {}
for record in remaining_nulls:
    uniq_id = record["UNIQ_ID"]
    desc = record["Description"]
    existing_up = record["UPKh_m_d"]

    if desc == "None" or desc is None:
        continue

    material = DESCRIPTION_TO_MATERIAL.get(desc, None)
    lit_gm = None
    if material is not None:
        lit_gm = geometric_mean_range(*LITERATURE_RANGES[material])

    if existing_up is None or existing_up == CONFIG["lake_value"]:
        if lit_gm is not None:
            up_val = lit_gm
            mid_val = lit_gm
        else:
            continue
    elif lit_gm is not None and existing_up > 0:
        lower = lit_gm * (1 - CONFIG["tolerance"])
        upper = lit_gm * (1 + CONFIG["tolerance"])
        if lower <= existing_up <= upper:
            up_val = lit_gm
            mid_val = lit_gm
        else:
            up_val = existing_up
            mid_val = existing_up
    else:
        up_val = existing_up
        mid_val = existing_up

    lit_assignments[uniq_id] = {"UPKh_krig": up_val, "MidKh_krig": mid_val}

with arcpy.da.UpdateCursor(poly_3174, ["UNIQ_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
    for row in cursor:
        uniq_id = row[0]
        if row[1] is None and uniq_id in lit_assignments:
            row[1] = lit_assignments[uniq_id]["UPKh_krig"]
            row[2] = lit_assignments[uniq_id]["UPKh_krig"]
            cursor.updateRow(row)

print("Applying manual conductivity fixes...")
fix_ids = {
    12: 8.64,
}

with arcpy.da.UpdateCursor(poly_3174, ["UNIQ_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
    for row in cursor:
        uniq_id = row[0]
        if uniq_id in fix_ids:
            row[1] = fix_ids[uniq_id]
            row[2] = fix_ids[uniq_id]
            cursor.updateRow(row)

print("Assigning lake-value conductivity to polygons that are primarily inside the lakes...")
arcpy.management.RepairGeometry(poly_3174)
arcpy.management.RepairGeometry(CONFIG["lake_shp"])

lake_clipped_raw = os.path.join(CONFIG["gdb"], "temp_lake_clip_raw")
delete_if_exists(lake_clipped_raw)
arcpy.analysis.Clip(poly_3174, CONFIG["lake_shp"], lake_clipped_raw)

if "orig_areas" not in globals() or len(orig_areas) == 0:
    orig_areas = {}
    with arcpy.da.SearchCursor(poly_3174, ["UNIQ_ID", "SHAPE@AREA"]) as cursor:
        for uniq_id, area in cursor:
            orig_areas[uniq_id] = area

oid_field_lk = arcpy.Describe(lake_clipped_raw).OIDFieldName
lake_fragments = defaultdict(list)
with arcpy.da.SearchCursor(lake_clipped_raw, [oid_field_lk, "UNIQ_ID", "SHAPE@AREA"]) as cursor:
    for oid, uniq_id, area in cursor:
        lake_fragments[uniq_id].append((oid, area))

lake_uniq_ids = set()
for uniq_id, fragments in lake_fragments.items():
    total_clip_area = sum(area for _, area in fragments)
    orig_area = orig_areas.get(uniq_id, 0)
    if orig_area > 0 and (total_clip_area / orig_area) >= CONFIG["min_lake_overlap_fraction"]:
        lake_uniq_ids.add(uniq_id)

with arcpy.da.UpdateCursor(poly_3174, ["UNIQ_ID", "UPKh_krig", "MidKh_krig"]) as cursor:
    for row in cursor:
        if row[0] in lake_uniq_ids:
            row[1] = CONFIG["lake_value"]
            row[2] = CONFIG["lake_value"]
            cursor.updateRow(row)

print(f"Fuzzy-matched polygons     : {len(fuzzy_assignments)}")
print(f"Literature/existing fills  : {len(lit_assignments)}")
print(f"Lake-dominant polygons set : {len(lake_uniq_ids)}")

In [ ]:
# ------------------------------------------------------------------
# STAGE 5 — REPORTS, TABLE EXPORTS, MAPS, AND FINAL SHAPEFILE
# ------------------------------------------------------------------
print("Creating change-tracking table...")
michigan_uniq_ids = set(uniq_lookup.keys()) if "uniq_lookup" in globals() else set()
fuzzy_uniq_ids = set(fuzzy_assignments.keys()) if "fuzzy_assignments" in globals() else set()
lit_uniq_ids = set(lit_assignments.keys()) if "lit_assignments" in globals() else set()
fix_uniq_ids = set(fix_ids.keys()) if "fix_ids" in globals() else set()

records = []
fields = [
    "UNIQ_ID", "POLYID", "Description",
    "UPKh_krig", "MidKh_krig",
    "UPKh_m_d", "MidKh_m_d",
]

with arcpy.da.SearchCursor(poly_3174, fields) as cursor:
    for uniq_id, polyid, desc, up_krig, mid_krig, up_md, mid_md in cursor:
        if up_krig is None:
            source = "NULL"
        elif uniq_id in fix_uniq_ids:
            source = "Manual fix"
        elif uniq_id in michigan_uniq_ids:
            source = "Michigan kriging"
        elif uniq_id in fuzzy_uniq_ids:
            source = "nearestMichigan match"
        elif uniq_id in lit_uniq_ids:
            assignment = lit_assignments[uniq_id]
            if up_md is not None and up_md != CONFIG["lake_value"] and assignment.get("UPKh_krig") is not None and abs(assignment["UPKh_krig"] - up_md) < 0.001:
                source = "Existing UPKh_m_d"
            else:
                source = "Literature"
        else:
            source = "Unknown"

        if source in ("Michigan kriging", "nearestMichigan match"):
            source_group = "Michigan_based"
        elif source in ("Literature", "Existing UPKh_m_d"):
            source_group = "Literature_Existing"
        elif source == "Manual fix":
            source_group = "Manual_fix"
        elif source == "NULL":
            source_group = "NULL"
        else:
            source_group = "Other"

        if up_krig is not None and up_md is not None:
            changed = abs(up_krig - up_md) > 0.001
        elif up_krig is None and up_md is None:
            changed = False
        else:
            changed = True

        if mid_krig is not None and mid_md is not None:
            mid_changed = abs(mid_krig - mid_md) > 0.001
        elif mid_krig is None and mid_md is None:
            mid_changed = False
        else:
            mid_changed = True

        records.append(
            {
                "UNIQ_ID": uniq_id,
                "POLYID": polyid,
                "Description": desc,
                "UPKh_m_d_original": up_md,
                "MidKh_m_d_original": mid_md,
                "UPKh_krig_new": up_krig,
                "MidKh_krig_new": mid_krig,
                "Source": source,
                "Source_Group": source_group,
                "Changed": changed,
                "Mid_Changed": mid_changed,
            }
        )

df = pd.DataFrame(records).sort_values(
    by=["Source_Group", "Source", "Changed", "UNIQ_ID"],
    ascending=[True, True, False, True],
).reset_index(drop=True)

df_michigan = df[df["Source"].isin(["Michigan kriging", "nearestMichigan match"])].copy()
df_lit_exist = df[df["Source"].isin(["Literature", "Existing UPKh_m_d"])].copy()
df_changed = df[df["Changed"]].copy()
df_unchanged = df[~df["Changed"]].copy()
df_other = df[
    ~df["Source"].isin(
        ["Michigan kriging", "nearestMichigan match", "Literature", "Existing UPKh_m_d"]
    )
].copy()

csv_all = os.path.join(CONFIG["out_dir"], "HK_change_tracking_all.csv")
csv_mich = os.path.join(CONFIG["out_dir"], "HK_change_tracking_michigan_based.csv")
csv_lit = os.path.join(CONFIG["out_dir"], "HK_change_tracking_literature_existing.csv")
xlsx_path = os.path.join(CONFIG["out_dir"], "HK_change_tracking.xlsx")

df.to_csv(csv_all, index=False)
df_michigan.to_csv(csv_mich, index=False)
df_lit_exist.to_csv(csv_lit, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="All_Changes")
    df_michigan.to_excel(writer, index=False, sheet_name="Michigan_based")
    df_lit_exist.to_excel(writer, index=False, sheet_name="Literature_Existing")
    df_changed.to_excel(writer, index=False, sheet_name="Changed_only")
    df_unchanged.to_excel(writer, index=False, sheet_name="Unchanged_only")
    df_other.to_excel(writer, index=False, sheet_name="Other_NULL_Manual")

print("Change-tracking exports complete.")
print(f"  CSV (all)               : {csv_all}")
print(f"  CSV (Michigan-based)    : {csv_mich}")
print(f"  CSV (Literature/existing): {csv_lit}")
print(f"  Excel workbook          : {xlsx_path}")

print("Reading polygon geometries for comparison plots...")
plot_records = []
with arcpy.da.SearchCursor(
    poly_3174,
    ["UNIQ_ID", "SHAPE@", "UPKh_m_d", "UPKh_krig", "MidKh_m_d", "MidKh_krig"],
) as cursor:
    for uniq_id, shape, up_md, up_kr, mid_md, mid_kr in cursor:
        poly_paths = []
        for part_index in range(shape.partCount):
            part = shape.getPart(part_index)
            coords = []
            for point in part:
                if point is None:
                    break
                coords.append((point.X, point.Y))
            if len(coords) > 2:
                poly_paths.append(coords)

        plot_records.append(
            {
                "UNIQ_ID": uniq_id,
                "paths": poly_paths,
                "UPKh_m_d": up_md if up_md is not None and up_md > 0 else 0.001,
                "UPKh_krig": up_kr if up_kr is not None and up_kr > 0 else 0.001,
                "MidKh_m_d": mid_md if mid_md is not None and mid_md > 0 else 0.001,
                "MidKh_krig": mid_kr if mid_kr is not None and mid_kr > 0 else 0.001,
            }
        )

hk_cmap = LinearSegmentedColormap.from_list("hk_15class", CONFIG["hk_colors_15"], N=512)
n_classes = 15

up_bounds, _ = compute_quantile_bounds(plot_records, "UPKh_m_d", "UPKh_krig", n_classes)
mid_bounds, _ = compute_quantile_bounds(plot_records, "MidKh_m_d", "MidKh_krig", n_classes)

fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(30, 14), facecolor="white")
fig1.subplots_adjust(wspace=0.12, right=0.86, left=0.05, top=0.90, bottom=0.06)
pc1, _ = plot_hk_map_quantile(ax1, plot_records, "UPKh_m_d", "UPKh_m_d (Original)", up_bounds, hk_cmap)
pc2, _ = plot_hk_map_quantile(ax2, plot_records, "UPKh_krig", "UPKh_krig (New — Kriging + Literature)", up_bounds, hk_cmap)

cbar_ax = fig1.add_axes([0.88, 0.06, 0.018, 0.82])
cb = fig1.colorbar(pc2, cax=cbar_ax, spacing="uniform")
cb.set_label("UPKh (m/d) — Quantile Classes", fontsize=12, color="black")
cb.set_ticks(up_bounds)
cb.set_ticklabels([format_tick(value) for value in up_bounds])
cb.ax.tick_params(labelsize=8, colors="black")
fig1.suptitle("Upper HK Comparison: Original vs New (15-Class Quantile)", fontsize=20, fontweight="bold", y=0.96, color="black")
fig1_path = os.path.join(CONFIG["out_dir"], "UPKh_comparison_quantile.png")
fig1.savefig(fig1_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(30, 14), facecolor="white")
fig2.subplots_adjust(wspace=0.12, right=0.86, left=0.05, top=0.90, bottom=0.06)
pc3, _ = plot_hk_map_quantile(ax3, plot_records, "MidKh_m_d", "MidKh_m_d (Original)", mid_bounds, hk_cmap)
pc4, _ = plot_hk_map_quantile(ax4, plot_records, "MidKh_krig", "MidKh_krig (New — Kriging + Literature)", mid_bounds, hk_cmap)

cbar_ax2 = fig2.add_axes([0.88, 0.06, 0.018, 0.82])
cb2 = fig2.colorbar(pc4, cax=cbar_ax2, spacing="uniform")
cb2.set_label("MidKh (m/d) — Quantile Classes", fontsize=12, color="black")
cb2.set_ticks(mid_bounds)
cb2.set_ticklabels([format_tick(value) for value in mid_bounds])
cb2.ax.tick_params(labelsize=8, colors="black")
fig2.suptitle("Mid HK Comparison: Original vs New (15-Class Quantile)", fontsize=20, fontweight="bold", y=0.96, color="black")
fig2_path = os.path.join(CONFIG["out_dir"], "MidKh_comparison_quantile.png")
fig2.savefig(fig2_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

fig3, (ax5, ax6) = plt.subplots(1, 2, figsize=(18, 8), facecolor="white")
up_old = [record["UPKh_m_d"] for record in plot_records if record["UPKh_m_d"] > 0.001 and record["UPKh_krig"] > 0.001]
up_new = [record["UPKh_krig"] for record in plot_records if record["UPKh_m_d"] > 0.001 and record["UPKh_krig"] > 0.001]
mid_old = [record["MidKh_m_d"] for record in plot_records if record["MidKh_m_d"] > 0.001 and record["MidKh_krig"] > 0.001]
mid_new = [record["MidKh_krig"] for record in plot_records if record["MidKh_m_d"] > 0.001 and record["MidKh_krig"] > 0.001]

ax5.scatter(up_old, up_new, c="#FB9F3A", edgecolor="white", linewidth=0.3, s=65, alpha=0.85)
lims = [min(min(up_old), min(up_new)) * 0.3, max(max(up_old), max(up_new)) * 3]
ax5.plot(lims, lims, "--", color="#FF4444", linewidth=1.5, label="1:1 line")
ax5.set_xscale("log")
ax5.set_yscale("log")
ax5.set_xlim(lims)
ax5.set_ylim(lims)
ax5.set_xlabel("UPKh_m_d (Original, m/d)")
ax5.set_ylabel("UPKh_krig (New, m/d)")
ax5.set_title("UPKh: Original vs New")
ax5.legend()
ax5.grid(True, alpha=0.15, which="both", color="gray")

ax6.scatter(mid_old, mid_new, c="#7E03A8", edgecolor="white", linewidth=0.3, s=65, alpha=0.85)
lims2 = [min(min(mid_old), min(mid_new)) * 0.3, max(max(mid_old), max(mid_new)) * 3]
ax6.plot(lims2, lims2, "--", color="#FF4444", linewidth=1.5, label="1:1 line")
ax6.set_xscale("log")
ax6.set_yscale("log")
ax6.set_xlim(lims2)
ax6.set_ylim(lims2)
ax6.set_xlabel("MidKh_m_d (Original, m/d)")
ax6.set_ylabel("MidKh_krig (New, m/d)")
ax6.set_title("MidKh: Original vs New")
ax6.legend()
ax6.grid(True, alpha=0.15, which="both", color="gray")

plt.tight_layout()
fig3_path = os.path.join(CONFIG["out_dir"], "HK_scatter_comparison.png")
fig3.savefig(fig3_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("Exporting final polygon shapefile...")
delete_if_exists(CONFIG["final_polygon_path"])
arcpy.conversion.FeatureClassToFeatureClass(poly_3174, CONFIG["out_dir"], CONFIG["final_polygon_name"])
print(f"Final polygon exported to: {CONFIG['final_polygon_path']}")

In [ ]:
# ------------------------------------------------------------------
# STAGE 6 — RASTERIZE FINAL HK POLYGONS TO A 5-BAND GEOTIFF
# ------------------------------------------------------------------
print("Rasterizing the final HK polygons to a 5-band 1000 m GeoTIFF...")
tmp_dir = os.path.join(CONFIG["out_dir"], "_tmp_hk5")
os.makedirs(tmp_dir, exist_ok=True)

arcpy.env.snapRaster = CONFIG["template_raster"]
arcpy.env.cellSize = CONFIG["template_raster"]
arcpy.env.extent = CONFIG["template_raster"]
arcpy.env.outputCoordinateSystem = arcpy.Describe(CONFIG["template_raster"]).spatialReference
arcpy.env.overwriteOutput = True

zone_1000m_path = os.path.join(tmp_dir, "zone_1000m.tif")
delete_if_exists(zone_1000m_path)

template_desc = arcpy.Describe(CONFIG["template_raster"])
arcpy.management.ProjectRaster(
    in_raster=CONFIG["water_mask_raster"],
    out_raster=zone_1000m_path,
    out_coor_system=template_desc.spatialReference,
    resampling_type="NEAREST",
    cell_size=template_desc.meanCellWidth,
)
zone_ras = Raster(zone_1000m_path)

band_rasters = []
for field_name in CONFIG["fields_5_layers_final"]:
    out_raster = os.path.join(tmp_dir, f"{field_name}.tif")
    filled_raster = os.path.join(tmp_dir, f"{field_name}_filled.tif")
    delete_if_exists(out_raster)
    delete_if_exists(filled_raster)

    arcpy.conversion.PolygonToRaster(
        in_features=poly_3174,
        value_field=field_name,
        out_rasterdataset=out_raster,
        cell_assignment="MAXIMUM_AREA",
        priority_field="NONE",
        cellsize=arcpy.env.cellSize,
    )

    band = Raster(out_raster)
    band_clean = SetNull(band == 0, band)
    band_domain = Con(zone_ras == 1, band_clean)

    mask_domain = Con(~IsNull(band_domain), 1)
    final = Nibble(
        in_raster=band_domain,
        in_mask_raster=mask_domain,
        nibble_values="DATA_ONLY",
        nibble_nodata="PROCESS_NODATA",
    )
    final = Con(zone_ras == 1, final)
    final.save(filled_raster)
    band_rasters.append(filled_raster)

delete_if_exists(CONFIG["multiband_hk_path"])
arcpy.management.CompositeBands(band_rasters, CONFIG["multiband_hk_path"])

arcpy.env.snapRaster = None
arcpy.env.cellSize = None
arcpy.env.extent = None
arcpy.env.outputCoordinateSystem = None

print(f"5-band HK raster saved to: {CONFIG['multiband_hk_path']}")

In [ ]:
# ------------------------------------------------------------------
# STAGE 7 — PLOT THE FINAL 5-BAND HK RASTER
# ------------------------------------------------------------------
print("Plotting the final 5-band HK raster...")
hk_cmap = LinearSegmentedColormap.from_list("hk_15class", CONFIG["hk_colors_15"], N=512)
band_names = CONFIG["fields_5_layers_final"]
n_classes = 15

with rasterio.open(CONFIG["multiband_hk_path"]) as src:
    nodata = src.nodata
    arrays = []
    for band_index in range(1, src.count + 1):
        array = src.read(band_index).astype(np.float32)
        if nodata is not None:
            array = np.where(array == nodata, np.nan, array)
        arrays.append(array)
    bounds_extent = src.bounds
    extent = [bounds_extent.left, bounds_extent.right, bounds_extent.bottom, bounds_extent.top]

fig, axes = plt.subplots(1, 5, figsize=(28, 7), facecolor="white")

for index, (array, name) in enumerate(zip(arrays, band_names)):
    ax = axes[index]
    finite = np.isfinite(array) & (array > 0)

    if finite.sum() == 0:
        ax.set_title(f"{name}\n(no data)", fontsize=10)
        ax.axis("off")
        continue

    values = array[finite]
    unique_vals = np.unique(values)

    if len(unique_vals) == 1:
        im = ax.imshow(
            array,
            extent=extent,
            origin="upper",
            cmap=hk_cmap,
            vmin=unique_vals[0] * 0.5,
            vmax=unique_vals[0] * 1.5,
        )
        ax.set_title(f"{name}\n(constant = {format_tick(unique_vals[0])})", fontsize=10, color="black")
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
        cb.set_label("HK (m/day)", fontsize=9)
    else:
        qbounds = np.percentile(values, np.linspace(0, 100, n_classes + 1))
        qbounds = np.unique(qbounds)
        norm = BoundaryNorm(qbounds, ncolors=hk_cmap.N, clip=True)
        im = ax.imshow(array, extent=extent, origin="upper", cmap=hk_cmap, norm=norm)
        ax.set_title(name, fontsize=11, fontweight="bold", color="black")

        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, spacing="uniform")
        cb.set_label("HK (m/day)", fontsize=9)
        cb.set_ticks(qbounds)
        cb.set_ticklabels([format_tick(value) for value in qbounds])
        cb.ax.tick_params(labelsize=7)

    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("HK 5-Band Raster — Quantile Classes (15)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(CONFIG["multiband_plot_path"], dpi=200, bbox_inches="tight", facecolor="white")
print(f"Final 5-band raster plot saved to: {CONFIG['multiband_plot_path']}")
plt.show()